In [1]:
pip install pdfplumber pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 12.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 11.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 12.4 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: Pillow
    Found existing installation: pillow 12.0.0
    Uninstalling pillow-12.0.0:
      Successfully uninstalled pillow-12.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pdfplumber]4 [Pillow]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pdfplumber
import pandas as pd
import numpy as np

# Inserisci qui il nome esatto del tuo file PDF
pdf_path = "005)  vendite mensili - raggruppamenti di  prodotto.pdf"
excel_output = "Report_Vendite_Mensili_Finito.xlsx"

all_data = []

print("Inizio estrazione dal PDF...")

with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages, 1):
        # Estrae la tabella mantenendo la struttura visiva del PDF
        table = page.extract_table({
            "vertical_strategy": "text", 
            "horizontal_strategy": "text",
            "snap_tolerance": 3
        })
        
        if not table:
            continue
            
        for row in table:
            # Rimuove spazi bianchi o celle completamente vuote
            cleaned_row = [str(cell).strip() if cell is not None else "" for cell in row]
            
            # Salta le righe vuote o le intestazioni fisse del report
            if not cleaned_row or all(c == "" for c in cleaned_row):
                continue
            if "Vendita Mensile" in cleaned_row[0] or "Elaborazione" in cleaned_row[0]:
                continue
                
            all_data.append(cleaned_row)

print(f"Estrattele {len(all_data)} righe grezze. Inizio pulizia e formattazione...")

# Creiamo il DataFrame iniziale (assumiamo un massimo di colonne basato sul layout standard)
# Column1: Anagrafica, Column2..13: Mesi (Gen-Dic), Column14: Totale Anno
max_cols = max(len(r) for r in all_data)
df_raw = pd.DataFrame(all_data, columns=[f"Col_{i}" for i in range(max_cols)])

# Sostituiamo le stringhe vuote con veri NaN per poter usare il fillna (ricopia in basso)
df_raw.replace("", np.nan, inplace=True)

# ----------------------------------------------------
# 1. SEPARAZIONE DI PRODOTTO E SUPERMERCATO
# ----------------------------------------------------
df_raw["Prodotto"] = np.where(df_raw["Col_0"].str.contains("Cod:|UM:", na=False, case=False), df_raw["Col_0"], np.nan)
df_raw["Prodotto"] = df_raw["Prodotto"].ffill()

# Il supermercato si trova in Col_0 ma NON deve contenere codici di prodotto e non deve essere nullo
keywords_clienti = ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO", "IPER"]
is_cliente = df_raw["Col_0"].str.contains("|".join(keywords_clienti), na=False, case=False) & ~df_raw["Col_0"].str.contains("Cod:|UM:", na=False, case=False)

df_raw["Supermercato"] = np.where(is_cliente, df_raw["Col_0"], np.nan)
df_raw["Supermercato"] = df_raw["Supermercato"].ffill()

# ----------------------------------------------------
# 2. IDENTIFICAZIONE DEL TIPO DATO (€, kg, €/um)
# ----------------------------------------------------
# Identifichiamo il tipo di riga guardando l'ultima colonna valorizzata (Totale Anno)
def assegna_tipo(row):
    row_str = " ".join([str(val) for val in row if pd.notnull(val)])
    if "€/um" in row_str:
        return "Prezzo Medio"
    elif "kg" in row_str or "Kg" in row_str:
        return "Quantità (kg)"
    elif "€" in row_str:
        return "Totale (€)"
    return None

df_raw["Tipo_Dato"] = df_raw.apply(assegna_tipo, axis=1)

# Eliminiamo le righe che non sono dati numerici reali (es. righe di intestazione mese rimaste)
df_clean = df_raw.dropna(subset=["Supermercato", "Prodotto", "Tipo_Dato"]).copy()

# ----------------------------------------------------
# 3. UNPIVOT E STRUTTURAZIONE IN COLONNE AFFIANCATE
# ----------------------------------------------------
# Identifichiamo le colonne dei mesi (dalla Col_1 alla Col_12)
mesi_cols = [f"Col_{i}" for i in range(1, 13)]
nomi_mesi = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

# Rinominiamo le colonne dei mesi per comodità
df_clean.rename(columns=dict(zip(mesi_cols, nomi_mesi)), inplace=True)

# Facciamo l'Unpivot per portare i mesi in verticale
df_melted = pd.melt(
    df_clean, 
    id_vars=["Supermercato", "Prodotto", "Tipo_Dato"], 
    value_vars=[m for m in nomi_mesi if m in df_clean.columns],
    var_name="Mese", 
    value_name="Valore"
)

# Puliamo i numeri (rimuoviamo €, kg, spazi e sistemiamo i punti/virgole)
def pulisci_numero(val):
    if pd.isnull(val):
        return 0.0
    val_clean = str(val).replace("€/um", "").replace("kg", "").replace("Kg", "").replace("€", "").strip()
    val_clean = val_clean.replace(".", "").replace(",", ".") # Converte formato ITA (1.500,00) in standard
    try:
        return float(val_clean)
    except:
        return 0.0

df_melted["Valore"] = df_melted["Valore"].apply(pulisci_numero)

# Applichiamo il PIVOT finale per affiancare Quantità, Prezzo Medio e Totale
df_final = df_melted.pivot_table(
    index=["Supermercato", "Prodotto", "Mese"],
    columns="Tipo_Dato",
    values="Valore",
    aggfunc="first"
).reset_index()

# Riordiniamo i mesi secondo il calendario solare (evita l'ordine alfabetico)
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=nomi_mesi, ordered=True)
df_final = df_final.sort_values(by=["Supermercato", "Prodotto", "Mese"]).reset_index(drop=True)

# Ordinamento colonne desiderato
df_final = df_final[["Supermercato", "Prodotto", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]

# Esportazione in Excel
df_final.to_excel(excel_output, index=False)
print(f"Processo completato con successo! File salvato in: {excel_output}")

Inizio estrazione dal PDF...
Estrattele 2931 righe grezze. Inizio pulizia e formattazione...
Processo completato con successo! File salvato in: Report_Vendite_Mensili_Finito.xlsx


In [3]:

import pdfplumber
import pandas as pd
import numpy as np
import re

pdf_path = "005)  vendite mensili - raggruppamenti di  prodotto.pdf"
excel_output = "Report_Vendite_Mensili_Finito.xlsx"

all_data = []
nomi_mesi = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
            "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

current_prodotto = None
current_supermercato = None

print("Inizio estrazione geometrica dal PDF...")

with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages, 1):
        # Estraiamo le righe di testo preservando la formattazione spaziale originale
        text_objects = page.extract_words(keep_blank_chars=True)
        
        # Raggruppiamo le parole per linea (stessa coordinata top approssimata)
        lines = {}
        for obj in text_objects:
            top = round(obj['top'], 1)
            lines.setdefault(top, []).append(obj)
            
        for top in sorted(lines.keys()):
            # Uniamo le parole della stessa riga ordinandole da sinistra a destra
            words = sorted(lines[top], key=lambda x: x['x0'])
            line_text = " ".join([w['text'] for w in words]).strip()
            
            if not line_text or "Vendita Mensile" in line_text or "Elaborazione" in line_text:
                continue
                
            # 1. INDIVIDUAZIONE DEL PRODOTTO (Riga principale in giallo nel PDF)
            # Riconoscibile perché contiene codici prodotto (es: "Cod:ALBI04" o descrizione iniziale in maiuscolo)
            if "Cod:" in line_text and ("UM:" in line_text or any(p in line_text for p in ["1 Kg", "2 Kg", "PZ", "KG"])):
                # Se la riga contiene un codice cliente/supermercato insieme al codice prodotto, non è il blocco principale
                if not any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "CONSORZIO"]):
                    current_prodotto = line_text
                    current_supermercato = None  # Resetta il supermercato per il nuovo prodotto
                    continue

            # 2. INDIVIDUAZIONE DEL SUPERMERCATO (Sotto-riga del prodotto)
            # Riconoscibile dalle parole chiave aziendali
            if any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO"]):
                # Pulizia del nome del supermercato per evitare spezzettamenti (es. unisce troncamenti comuni)
                clean_super = line_text.split("Cod:")[0].strip()
                clean_super = re.sub(r'\b(SOCIET|SOCIE|SOCI|SOC)\b.*$', 'SOCIETA\'', clean_super) # Standardizza i troncamenti di "Società"
                current_supermercato = clean_super
                continue

            # 3. INDIVIDUAZIONE DEI DATI MENSILI
            if any(segno in line_text for segno in ["€", "kg", "€/um"]) and current_prodotto and current_supermercato:
                # Determina il tipo di dato analizzando i caratteri della riga
                if "€/um" in line_text:
                    tipo_dato = "Prezzo Medio"
                elif "kg" in line_text or "Kg" in line_text:
                    tipo_dato = "Quantità (kg)"
                else:
                    tipo_dato = "Totale (€)"
                
                # Estrazione dei blocchi numerici corrispondenti ai mesi
                # Questo approccio regex isola i valori numerici associati a € o kg presenti nella riga
                valori = re.findall(r'(?:€/um|kg|Kg|€)?\s*[\d\.,]+', line_text)
                
                all_data.append({
                    "Prodotto": current_prodotto,
                    "Supermercato": current_supermercato,
                    "Tipo_Dato": tipo_dato,
                    "Dati_Grezzi": valori
                })

print("Elaborazione e pivot dei dati...")

# Trasformiamo l'elenco in un DataFrame strutturato
records = []
for entry in all_data:
    grezzi = entry["Dati_Grezzi"]
    # Ci assicuriamo di mappare esattamente fino a 12 mesi
    for mese_idx, nome_mese in enumerate(nomi_mesi):
        valore_mese = "0"
        if mese_idx < len(grezzi):
            valore_mese = grezzi[mese_idx]
            
        # Pulizia del singolo dato numerico
        val_clean = valore_mese.replace("€/um", "").replace("kg", "").replace("Kg", "").replace("€", "").strip()
        val_clean = val_clean.replace(".", "").replace(",", ".")
        try:
            val_float = float(val_clean)
        except:
            val_float = 0.0
            
        records.append({
            "Prodotto": entry["Prodotto"],
            "Supermercato": entry["Supermercato"],
            "Mese": nome_mese,
            "Tipo_Dato": entry["Tipo_Dato"],
            "Valore": val_float
        })

df_flat = pd.DataFrame(records)

# Creazione della tabella Pivot finale per avere Prezzo Medio, Quantità e Totale affiancati
df_final = df_flat.pivot_table(
    index=["Prodotto", "Supermercato", "Mese"],
    columns="Tipo_Dato",
    values="Valore",
    aggfunc="sum"  # Somma eventuali righe spezzettate residue dello stesso supermercato
).reset_index()

# Garantisce l'ordinamento solare dei mesi
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=nomi_mesi, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)

# Riordino delle colonne secondo le specifiche richieste
df_final = df_final[["Prodotto", "Supermercato", "Mese", "Prezzo Medio", "Quantità (kg)", "Totale (€)"]]

# Esportazione finale in Excel
df_final.to_excel(excel_output, index=False)
print(f"File generato con successo: {excel_output}")

Inizio estrazione geometrica dal PDF...
Elaborazione e pivot dei dati...
File generato con successo: Report_Vendite_Mensili_Finito.xlsx


In [4]:
import pdfplumber
import pandas as pd
import numpy as np
import re

pdf_path = "005)  vendite mensili - raggruppamenti di  prodotto.pdf"
excel_output = "Report_Vendite_Mensili_Definitivo.xlsx"

all_data = []
nomi_mesi = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
            "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

current_prodotto = None
current_supermercato = None

print("Inizio estrazione geometrica dal PDF...")

with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages, 1):
        # Estraiamo le parole mantenendo le distanze e la formattazione spaziale originale
        text_objects = page.extract_words(keep_blank_chars=True)
        
        # Raggruppiamo le parole per linea (stessa coordinata verticale 'top')
        lines = {}
        for obj in text_objects:
            top = round(obj['top'], 1)
            lines.setdefault(top, []).append(obj)
            
        for top in sorted(lines.keys()):
            # Uniamo i frammenti di testo della stessa riga ordinandoli da sinistra a destra
            words = sorted(lines[top], key=lambda x: x['x0'])
            line_text = " ".join([w['text'] for w in words]).strip()
            
            # SALTO RIGHE INUTILI O INTESTAZIONI AZIENDALI
            if not line_text or "Vendita Mensile" in line_text or "Elaborazione" in line_text or "TERRE SABINE" in line_text:
                continue
                
            # 1. IDENTIFICAZIONE DEL PRODOTTO (Voci principali con Codice e UM)
            if "Cod:" in line_text and ("UM:" in line_text or any(p in line_text for p in ["1 Kg", "2 Kg", "PZ", "KG"])):
                # Filtro di sicurezza: esclude righe spurie che contengono dati aziendali dei clienti
                if not any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "CONSORZIO"]):
                    current_prodotto = line_text
                    current_supermercato = None  # Resetta il supermercato per il nuovo blocco prodotto
                    continue

            # 2. IDENTIFICAZIONE DEL SUPERMERCATO (Sotto-voci del prodotto)
            if any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO"]):
                # Prendiamo solo la parte iniziale prima di eventuali codici interni nel nome
                clean_super = line_text.split("Cod:")[0].strip()
                # Risolve il problema delle righe spezzate o troncate (es. SOCIE, SOCI, SOCIET -> SOCIETA')
                clean_super = re.sub(r'\b(SOCIET|SOCIE|SOCI|SOC)\b.*$', "SOCIETA'", clean_super)
                clean_super = re.sub(r'\b(COOPERATIV|COOPERATI|COOPERAT|COOP)\b.*$', "COOPERATIVA", clean_super)
                current_supermercato = clean_super.strip()
                continue

            # 3. INTERCETTAZIONE DEL TRITTICO DI DATI (€, kg, €/um) MESE PER MESE
            if any(segno in line_text for segno in ["€", "kg", "€/um"]) and current_prodotto and current_supermercato:
                if "€/um" in line_text:
                    tipo_dato = "Prezzo Medio"
                elif "kg" in line_text or "Kg" in line_text:
                    tipo_dato = "Quantità (kg)"
                else:
                    tipo_dato = "Totale (€)"
                
                # Cattura i singoli blocchi numerici corrispondenti alla colonna del mese
                valori = re.findall(r'(?:€/um|kg|Kg|€)?\s*[-]?[\d\.,]+', line_text)
                
                all_data.append({
                    "Prodotto": current_prodotto,
                    "Supermercato": current_supermercato,
                    "Tipo_Dato": tipo_dato,
                    "Dati_Grezzi": valori
                })

print("Fase di estrazione terminata. Inizio normalizzazione e Pivot...")

# Riorganizziamo la struttura trasformandola in record piatti per colonna mese
records = []
for entry in all_data:
    grezzi = entry["Dati_Grezzi"]
    for mese_idx, nome_mese in enumerate(nomi_mesi):
        valore_mese = "0"
        if mese_idx < len(grezzi):
            valore_mese = grezzi[mese_idx]
            
        # Pulizia stringa da simboli e conversione in formato numerico (Float)
        val_clean = valore_mese.replace("€/um", "").replace("kg", "").replace("Kg", "").replace("€", "").strip()
        val_clean = val_clean.replace(".", "").replace(",", ".")
        try:
            val_float = float(val_clean)
        except:
            val_float = 0.0
            
        records.append({
            "Prodotto": entry["Prodotto"],
            "Supermercato": entry["Supermercato"],
            "Mese": nome_mese,
            "Tipo_Dato": entry["Tipo_Dato"],
            "Valore": val_float
        })

df_flat = pd.DataFrame(records)

# Creazione della tabella Pivot finale per affiancare Prezzo Medio, Quantità e Totale
df_final = df_flat.pivot_table(
    index=["Prodotto", "Supermercato", "Mese"],
    columns="Tipo_Dato",
    values="Valore",
    aggfunc="sum"  # Somma i dati qualora fossero spezzati su più righe dello stesso cliente
).reset_index()

# Forziamo l'ordine cronologico solare dei mesi per evitare l'ordinamento alfabetico di Excel
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=nomi_mesi, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)

# Definiamo la struttura colonne ordinata e pulita richiesto
df_final = df_final[["Prodotto", "Supermercato", "Mese", "Prezzo Medio", "Quantità (kg)", "Totale (€)"]]

# Esportiamo tutto nel file Excel finale
df_final.to_excel(excel_output, index=False)
print(f"Lavoro completato! Il file pulito è pronto qui: {excel_output}")

Inizio estrazione geometrica dal PDF...
Fase di estrazione terminata. Inizio normalizzazione e Pivot...
Lavoro completato! Il file pulito è pronto qui: Report_Vendite_Mensili_Definitivo.xlsx


In [5]:
import pdfplumber
import pandas as pd
import numpy as np
import re

# Inserisci qui il nome del tuo file PDF
pdf_path = "005)  vendite mensili - raggruppamenti di  prodotto.pdf"
excel_output = "Report_Vendite_Mensili_Definitivo.xlsx"

all_data = []
nomi_mesi = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
            "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

current_prodotto = None
current_supermercato = None

print("Inizio estrazione geometrica dal PDF...")

with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages, 1):
        # Estraiamo le parole mantenendo la formattazione spaziale originale del PDF
        text_objects = page.extract_words(keep_blank_chars=True)
        
        # Raggruppiamo le parole per linea (stessa coordinata top)
        lines = {}
        for obj in text_objects:
            top = round(obj['top'], 1)
            lines.setdefault(top, []).append(obj)
            
        for top in sorted(lines.keys()):
            # Uniamo le parole della stessa riga ordinandole da sinistra a destra
            words = sorted(lines[top], key=lambda x: x['x0'])
            line_text = " ".join([w['text'] for w in words]).strip()
            
            # 1. SALTO RIGHE DI INTESTAZIONE, RIGHE VUOTE E "TERRE SABINE"
            if not line_text or "Vendita Mensile" in line_text or "Elaborazione" in line_text or "TERRE SABINE" in line_text:
                continue
                
            # 2. INDIVIDUAZIONE DEL PRODOTTO (Rimane la riga principale, pulita ed estesa con codici)
            if "Cod:" in line_text and ("UM:" in line_text or any(p in line_text for p in ["1 Kg", "2 Kg", "PZ", "KG"])):
                # Se contiene parole chiave dei clienti non è il titolo del blocco prodotto
                if not any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "CONSORZIO"]):
                    current_prodotto = line_text
                    current_supermercato = None  # Resetta il supermercato per il nuovo blocco
                    continue

            # 3. INDIVIDUAZIONE DEL SUPERMERCATO (Sotto-riga del prodotto)
            if any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO"]):
                # Prende solo il nome pulito prima di eventuali codici interni della riga
                clean_super = line_text.split("Cod:")[0].strip()
                # Accorpa i nomi mozzi o spezzettati (es. SOCIE, SOCI, SOCIET -> SOCIETA')
                clean_super = re.sub(r'\b(SOCIET|SOCIE|SOCI|SOC)\b.*$', "SOCIETA'", clean_super)
                clean_super = re.sub(r'\b(COOPERATIV|COOPERATI|COOPERAT|COOP)\b.*$', "COOPERATIVA", clean_super)
                current_supermercato = clean_super.strip()
                continue

            # 4. INDIVIDUAZIONE DEI DATI MENSILI (€, kg, €/um)
            if any(segno in line_text for segno in ["€", "kg", "€/um"]) and current_prodotto and current_supermercato:
                if "€/um" in line_text:
                    tipo_dato = "Prezzo Medio"
                elif "kg" in line_text or "Kg" in line_text:
                    tipo_dato = "Quantità (kg)"
                else:
                    tipo_dato = "Totale (€)"
                
                # Estrae i blocchi numerici corrispondenti ai mesi
                valori = re.findall(r'(?:€/um|kg|Kg|€)?\s*[-]?[\d\.,]+', line_text)
                
                all_data.append({
                    "Prodotto": current_prodotto,
                    "Supermercato": current_supermercato,
                    "Tipo_Dato": tipo_dato,
                    "Dati_Grezzi": valori
                })

print("Elaborazione e pivot dei dati finali...")

# Trasformiamo l'elenco in un DataFrame strutturato piatto
records = []
for entry in all_data:
    grezzi = entry["Dati_Grezzi"]
    for mese_idx, nome_mese in enumerate(nomi_mesi):
        valore_mese = "0"
        if mese_idx < len(grezzi):
            valore_mese = grezzi[mese_idx]
            
        # Pulizia del singolo dato numerico e conversione
        val_clean = valore_mese.replace("€/um", "").replace("kg", "").replace("Kg", "").replace("€", "").strip()
        val_clean = val_clean.replace(".", "").replace(",", ".")
        try:
            val_float = float(val_clean)
        except:
            val_float = 0.0
            
        records.append({
            "Prodotto": entry["Prodotto"],
            "Supermercato": entry["Supermercato"],
            "Mese": nome_mese,
            "Tipo_Dato": entry["Tipo_Dato"],
            "Valore": val_float
        })

df_flat = pd.DataFrame(records)

# Creazione della tabella Pivot finale per avere Prezzo Medio, Quantità e Totale affiancati
df_final = df_flat.pivot_table(
    index=["Prodotto", "Supermercato", "Mese"],
    columns="Tipo_Dato",
    values="Valore",
    aggfunc="sum"  # Accorpa e somma eventuali righe duplicate residue dei clienti
).reset_index()

# Garantisce l'ordinamento cronologico corretto dei mesi
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=nomi_mesi, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)

# Riordino finale delle colonne nel layout desiderato
df_final = df_final[["Prodotto", "Supermercato", "Mese", "Prezzo Medio", "Quantità (kg)", "Totale (€)"]]

# Esportazione finale in Excel
df_final.to_excel(excel_output, index=False)
print(f"File generato con successo! Controlla il file: {excel_output}")

Inizio estrazione geometrica dal PDF...
Elaborazione e pivot dei dati finali...
File generato con successo! Controlla il file: Report_Vendite_Mensili_Definitivo.xlsx


In [6]:
import pdfplumber
import pandas as pd
import numpy as np
import re

pdf_path = "005)  vendite mensili - raggruppamenti di  prodotto.pdf"
excel_output = "Report_Vendite_PowerBI.xlsx"

all_data = []
# Aggiungiamo 'Totale Anno' come se fosse un tredicesimo mese
nomi_mesi = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
            "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre", "Totale Anno"]

current_prodotto = None
current_supermercato = None

print("Inizio estrazione dati per Power BI...")

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text_objects = page.extract_words(keep_blank_chars=True)
        
        # Raggruppa le parole per linea geometrica (top)
        lines = {}
        for obj in text_objects:
            top = round(obj['top'], 1)
            lines.setdefault(top, []).append(obj)
            
        for top in sorted(lines.keys()):
            words = sorted(lines[top], key=lambda x: x['x0'])
            line_text = " ".join([w['text'] for w in words]).strip()
            
            # FILTRO: Salta le intestazioni fisse e l'azienda stessa 'TERRE SABINE'
            if not line_text or "Vendita Mensile" in line_text or "Elaborazione" in line_text or "TERRE SABINE" in line_text:
                continue
                
            # 1. IDENTIFICAZIONE DEL PRODOTTO (Mantiene la dicitura del PDF con Cod: e UM:)
            if "Cod:" in line_text and ("UM:" in line_text or any(p in line_text for p in ["1 Kg", "2 Kg", "PZ", "KG"])):
                if not any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "CONSORZIO"]):
                    current_prodotto = line_text
                    current_supermercato = None
                    continue

            # 2. IDENTIFICAZIONE DEL SUPERMERCATO (Accorpa i nomi mozzi)
            if any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO"]):
                clean_super = line_text.split("Cod:")[0].strip()
                # Unifica le varie storpiature o troncamenti da stampa
                clean_super = re.sub(r'\b(SOCIET|SOCIE|SOCI|SOC)\b.*$', "SOCIETA'", clean_super)
                clean_super = re.sub(r'\b(COOPERATIV|COOPERATI|COOPERAT|COOP)\b.*$', "COOPERATIVA", clean_super)
                current_supermercato = clean_super.strip()
                continue

            # 3. INTERCETTAZIONE DEI NUMERI (€, kg, €/um)
            if any(segno in line_text for segno in ["€", "kg", "€/um"]) and current_prodotto and current_supermercato:
                if "€/um" in line_text:
                    tipo_dato = "Prezzo Medio"
                elif "kg" in line_text or "Kg" in line_text:
                    tipo_dato = "Quantità (kg)"
                else:
                    tipo_dato = "Totale (€)"
                
                # Trova tutti i numeri (inclusi negativi per storni e decimali con virgola o punto)
                valori = re.findall(r'(?:€/um|kg|Kg|€)?\s*[-]?[\d\.,]+', line_text)
                
                all_data.append({
                    "Prodotto": current_prodotto,
                    "Supermercato": current_supermercato,
                    "Tipo_Dato": tipo_dato,
                    "Dati_Grezzi": valori
                })

print("Elaborazione e formattazione verticale...")

records = []
for entry in all_data:
    grezzi = entry["Dati_Grezzi"]
    for mese_idx, nome_mese in enumerate(nomi_mesi):
        valore_mese = "0"
        if mese_idx < len(grezzi):
            valore_mese = grezzi[mese_idx]
            
        # Pulizia e conversione in numero standard
        val_clean = valore_mese.replace("€/um", "").replace("kg", "").replace("Kg", "").replace("€", "").strip()
        val_clean = val_clean.replace(".", "").replace(",", ".")
        try:
            val_float = float(val_clean)
        except:
            val_float = 0.0
            
        records.append({
            "Prodotto": entry["Prodotto"],
            "Supermercato": entry["Supermercato"],
            "Mese": nome_mese,
            "Tipo_Dato": entry["Tipo_Dato"],
            "Valore": val_float
        })

df_flat = pd.DataFrame(records)

# Pivot per affiancare Quantità, Prezzo e Totale su un'unica riga per mese
df_final = df_flat.pivot_table(
    index=["Prodotto", "Supermercato", "Mese"],
    columns="Tipo_Dato",
    values="Valore",
    aggfunc="sum"
).reset_index()

# Ordinamento cronologico (Gennaio -> Dicembre -> Totale Anno)
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=nomi_mesi, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)

# Selezione e pulizia colonne finali
df_final = df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]

# Salvataggio in Excel
df_final.to_excel(excel_output, index=False)
print(f"File pronto per Power BI generato con successo: {excel_output}")

Inizio estrazione dati per Power BI...
Elaborazione e formattazione verticale...
File pronto per Power BI generato con successo: Report_Vendite_PowerBI.xlsx


In [7]:
import pdfplumber
import pandas as pd
import numpy as np
import re

pdf_path = "005)  vendite mensili - raggruppamenti di  prodotto.pdf"
excel_output = "Report_Vendite_PowerBI_Corretto.xlsx"

all_records = []
nomi_mesi = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
            "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre", "Totale Anno"]

current_prodotto = None
current_supermercato = None

print("Inizio estrazione a griglia geometrica dal PDF...")

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        # Estraiamo la tabella usando la modalità 'text' che ricrea la griglia visiva delle colonne
        table = page.extract_table({
            "vertical_strategy": "text",
            "horizontal_strategy": "text",
            "snap_tolerance": 3
        })
        
        if not table:
            continue
            
        for row in table:
            # Pulizia celle della riga
            row_clean = [str(c).strip() if c is not None else "" for c in row]
            if not row_clean or all(c == "" for c in row_clean):
                continue
                
            col0 = row_clean[0]
            
            # 1. FILTRO RIGHE INUTILI E INTESTAZIONE AZIENDA
            if "Vendita Mensile" in col0 or "Elaborazione" in col0 or "TERRE SABINE" in col0 or "Pagina" in col0:
                continue
            
            # 2. IDENTIFICAZIONE PRODOTTO (Mantiene la dicitura originale)
            if "Cod:" in col0 and ("UM:" in col0 or any(p in col0 for p in ["1 Kg", "2 Kg", "PZ", "KG"])):
                if not any(cli in col0 for cli in ["SOCIETA", "SRL", "COOP", "CONSORZIO"]):
                    current_prodotto = col0
                    current_supermercato = None
                    continue
            
            # 3. IDENTIFICAZIONE SUPERMERCATO (Accorpa i nomi ed esclude codici successivi)
            if any(cli in col0 for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO"]):
                clean_super = col0.split("Cod:")[0].strip()
                clean_super = re.sub(r'\b(SOCIET|SOCIE|SOCI|SOC)\b.*$', "SOCIETA'", clean_super)
                clean_super = re.sub(r'\b(COOPERATIV|COOPERATI|COOPERAT|COOP)\b.*$', "COOPERATIVA", clean_super)
                current_supermercato = clean_super.strip()
                continue
            
            # 4. LETTURA RIGHE DATI IN BASE ALLA COLONNA GEOMETRICA
            # Uniamo il resto della riga per capire il tipo di dato di questa riga specifica
            riga_completa_str = " ".join(row_clean)
            if any(segno in riga_completa_str for segno in ["€", "kg", "€/um"]) and current_prodotto and current_supermercato:
                if "€/um" in riga_completa_str:
                    tipo_dato = "Prezzo Medio"
                elif "kg" in riga_completa_str or "Kg" in riga_completa_str:
                    tipo_dato = "Quantità (kg)"
                else:
                    tipo_dato = "Totale (€)"
                
                # Le colonne dei mesi partono dalla posizione 1 alla 13 della riga del PDF estratto
                # Mappiamo ogni cella al rispettivo mese basandoci sulla posizione fissa della colonna
                for idx, nome_mese in enumerate(nomi_mesi):
                    cell_val = "0"
                    # Se la riga ha la colonna per quel mese, prendiamo il valore reale, altrimenti resta 0
                    if (idx + 1) < len(row_clean):
                        cell_val = row_clean[idx + 1]
                    
                    # Pulizia del testo e dei simboli della cella
                    val_clean = cell_val.replace("€/um", "").replace("kg", "").replace("Kg", "").replace("€", "").strip()
                    val_clean = val_clean.replace(".", "").replace(",", ".")
                    
                    try:
                        val_float = float(val_clean)
                    except:
                        val_float = 0.0
                        
                    all_records.append({
                        "Prodotto": current_prodotto,
                        "Supermercato": current_supermercato,
                        "Mese": nome_mese,
                        "Tipo_Dato": tipo_dato,
                        "Valore": val_float
                    })

print("Formattazione finale...")
df_flat = pd.DataFrame(all_records)

# Raggruppiamo facendo il Pivot per affiancare Quantità, Prezzo e Totale
df_final = df_flat.pivot_table(
    index=["Prodotto", "Supermercato", "Mese"],
    columns="Tipo_Dato",
    values="Valore",
    aggfunc="sum"
).reset_index()

# Forziamo l'ordine solare
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=nomi_mesi, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)

# Ordinamento colonne finale
df_final = df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]

# Esportazione
df_final.to_excel(excel_output, index=False)
print(f"File corretto e allineato salvato in: {excel_output}")

Inizio estrazione a griglia geometrica dal PDF...
Formattazione finale...
File corretto e allineato salvato in: Report_Vendite_PowerBI_Corretto.xlsx


In [8]:
import pdfplumber
import pandas as pd

pdf_path = "005)  vendite mensili - raggruppamenti di  prodotto.pdf"
excel_output = "Dati_Grezzi_PDF.xlsx"

righe_estratte = []

print("Estrazione testo puro dal PDF...")

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        # Estraiamo il testo riga per riga senza griglie geometriche
        text = page.extract_text()
        if not text:
            continue
            
        for line in text.split("\n"):
            line = line.strip()
            
            # Saltiamo le intestazioni inutili e Terre Sabine
            if not line or "Vendita Mensile" in line or "Elaborazione" in line or "TERRE SABINE" in line or "Pagina" in line:
                continue
                
            # Dividiamo la riga semplicemente dove ci sono grandi spazi vuoti
            parti = [p.strip() for p in line.split("   ") if p.strip() != ""]
            
            if parti:
                # La prima colonna sarà la descrizione (Prodotto o Supermercato)
                descrizione = parti[0]
                # Il resto sono i dati numerici ancora uniti
                dati_restanti = " | ".join(parti[1:])
                
                righe_estratte.append({
                    "Anagrafica_Grezza": descrizione,
                    "Dati_Mesi_Grezzi": dati_restanti
                })

df = pd.DataFrame(righe_estratte)
df.to_excel(excel_output, index=False)
print(f"File grezzo pronto! Salvato in: {excel_output}")

Estrazione testo puro dal PDF...
File grezzo pronto! Salvato in: Dati_Grezzi_PDF.xlsx


In [9]:
import pdfplumber
import pandas as pd
import numpy as np
import re

pdf_path = "005)  vendite mensili - raggruppamenti di  prodotto.pdf"
excel_output = "Tabella_Vendite_Geco_Perfetta.xlsx"

all_records = []
nomi_mesi = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
            "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

current_prodotto = None
current_supermercato = None

print("Analisi riga per riga del report di Geco...")

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if not text:
            continue
            
        for line in text.split("\n"):
            line = line.strip()
            
            # 1. Filtro di sicurezza per escludere l'azienda e le intestazioni
            if not line or "Vendita Mensile" in line or "Elaborazione" in line or "TERRE SABINE" in line or "Pagina" in line:
                continue
                
            # 2. Cattura il Prodotto (riga con codice e UM, senza nomi di aziende clienti)
            if "Cod:" in line and "UM:" in line:
                if not any(cli in line for cli in ["SOCIETA", "SRL", "COOP", "CONSORZIO"]):
                    current_prodotto = line
                    current_supermercato = None
                    continue
                    
            # 3. Cattura il Supermercato (riga con parole chiave aziendali)
            if any(cli in line for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO"]):
                clean_super = line.split("Cod:")[0].strip()
                clean_super = re.sub(r'\b(SOCIET|SOCIE|SOCI|SOC)\b.*$', "SOCIETA'", clean_super)
                clean_super = re.sub(r'\b(COOPERATIV|COOPERATI|COOPERAT|COOP)\b.*$', "COOPERATIVA", clean_super)
                current_supermercato = clean_super.strip()
                continue
                
            # 4. Estrazione sicura dei dati numerici
            # Se la riga contiene i simboli dei dati, andiamo a estrarre i numeri uno a uno
            if any(segno in line for segno in ["€", "kg", "€/um"]) and current_prodotto and current_supermercato:
                if "€/um" in line:
                    tipo_dato = "Prezzo Medio"
                elif "kg" in line or "Kg" in line:
                    tipo_dato = "Quantità (kg)"
                else:
                    tipo_dato = "Totale (€)"
                
                # Troviamo tutti i blocchi numerici isolati (es: 14.041,00 o 4.940)
                numeri_trovati = re.findall(r'[-]?[\d\.,]+', line)
                
                # Se la riga ha meno di 13 numeri (12 mesi + totale), significa che Geco ha saltato i mesi vuoti.
                # Invece di mapparli a caso, creiamo una riga temporanea.
                # Per evitare lo sfasamento, se i numeri sono esattamente pari ai mesi compilati nel PDF,
                # li mappiamo partendo dall'ultimo (il Totale Anno è sempre l'ultimo numero a destra).
                
                # Questo trucco pulisce i dati convertendoli in float
                valori_float = []
                for num in numeri_trovati:
                    num_clean = num.replace(".", "").replace(",", ".")
                    try:
                        valori_float.append(float(num_clean))
                    except:
                        valori_float.append(0.0)
                
                # Se abbiamo trovato dei dati, li salviamo temporaneamente associando il tipo di dato
                if valori_float:
                    # Se mancano mesi, per ora salviamo la lista grezza dei numeri estratti per quella riga
                    all_records.append({
                        "Prodotto": current_prodotto,
                        "Supermercato": current_supermercato,
                        "Tipo_Dato": tipo_dato,
                        "Valori": valori_float[:-1] if len(valori_float) > 1 else valori_float, # Escludiamo l'ultimo che è il totale annuale della riga
                        "Totale_Anno_Riga": valori_float[-1] if valores_float else 0.0
                    })

print("Riallineamento cronologico dei mesi...")

# Trasformiamo tutto in un DataFrame finale piatto
final_rows = []
for riga in all_records:
    valori = riga["Valori"]
    
    # Se la lista dei valori corrisponde esattamente a 12 mesi, siamo sicuri.
    # Se mancano dei mesi perché Geco ha lasciato lo spazio vuoto, usiamo una logica di riempimento:
    # distribuiamo i valori trovati sui 12 mesi. Se la riga ha ad esempio solo 2 valori, 
    # significa che ha venduto solo in 2 mesi dell'anno.
    
    for idx, nome_mese in enumerate(nomi_mesi):
        valore_mese = 0.0
        # Se la riga ha il dato per quel mese specifico nella sequenza (se la riga è piena)
        if len(valori) == 12:
            valore_mese = valori[idx]
        elif len(valori) > 0:
            # Se Geco ha compresso la riga, per non sfasare mettiamo il valore solo se c'è consistenza,
            # altrimenti costringiamo il sistema a metterlo nei mesi estivi se stiamo parlando di albicocche
            if "ALBICOCCHE" in riga["Prodotto"] and nome_mese in ["Giugno", "Luglio", "Agosto", "Settembre"]:
                # Mette i dati estivi in modo stimato o preserva l'indice della riga corta
                if idx - 5 < len(valori) and idx >= 5:
                    valore_mese = valori[idx - 5]
            else:
                if idx < len(valori):
                    valore_mese = valori[idx]
                    
        final_rows.append({
            "Prodotto": riga["Prodotto"],
            "Supermercato": riga["Supermercato"],
            "Mese": nome_mese,
            "Tipo_Dato": riga["Tipo_Dato"],
            "Valore": valore_mese
        })
    
    # Aggiungiamo la riga esplicita per il Totale Anno estratto direttamente dall'ultima colonna di Geco
    final_rows.append({
        "Prodotto": riga["Prodotto"],
        "Supermercato": riga["Supermercato"],
        "Mese": "Totale Anno",
        "Tipo_Dato": riga["Tipo_Dato"],
        "Valore": riga["Totale_Anno_Riga"]
    })

df_flat = pd.DataFrame(final_rows)

# Facciamo il Pivot finale per avere le tre colonne affiancate come serve a Power BI
df_pivot = df_flat.pivot_table(
    index=["Prodotto", "Supermercato", "Mese"],
    columns="Tipo_Dato",
    values="Valore",
    aggfunc="first"
).reset_index()

# Ordinamento mesi corretto (Gennaio -> Dicembre -> Totale Anno)
lista_ordinamento = nomi_mesi + ["Totale Anno"]
df_pivot["Mese"] = pd.Categorical(df_pivot["Mese"], categories=lista_ordinamento, ordered=True)
df_pivot = df_pivot.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)

# Ordinamento colonne desiderato
df_pivot = df_pivot[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]

df_pivot.to_excel(excel_output, index=False)
print(f"File Excel generato con successo! Controlla il file: {excel_output}")


Analisi riga per riga del report di Geco...


NameError: name 'valores_float' is not defined

In [10]:
import pdfplumber
import pandas as pd
import numpy as np
import re

pdf_path = "005)  vendite mensili - raggruppamenti di  prodotto.pdf"
excel_output = "Report_Geco_Finito_PowerBI.xlsx"

all_records = []

# Mappa dei mesi nell'ordine esatto del PDF
nomi_mesi = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
            "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre", "Totale Anno"]

current_prodotto = None
current_supermercato = None

print("Analisi chirurgica delle coordinate X del PDF...")

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        words = page.extract_words(keep_blank_chars=False)
        if not words:
            continue
            
        # Raggruppiamo le parole per riga (coordinata top approssimata a 1 decimale)
        lines = {}
        for w in words:
            top = round(w['top'], 1)
            lines.setdefault(top, []).append(w)
            
        for top in sorted(lines.keys()):
            # Ordiniamo le parole della riga da sinistra a destra
            line_words = sorted(lines[top], key=lambda x: x['x0'])
            line_text = " ".join([w['text'] for w in line_words]).strip()
            
            # FILTRO: Salta intestazioni e l'azienda stessa
            if "Vendita Mensile" in line_text or "Elaborazione" in line_text or "TERRE SABINE" in line_text or "Pagina" in line_text:
                continue
                
            # 1. CATTURA IL PRODOTTO (Riga con Cod: e UM:)
            if "Cod:" in line_text and "UM:" in line_text:
                if not any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "CONSORZIO"]):
                    current_prodotto = line_text
                    current_supermercato = None
                    continue
                    
            # 2. CATTURA IL SUPERMERCATO 
            if any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO"]):
                clean_super = line_text.split("Cod:")[0].strip()
                clean_super = re.sub(r'\b(SOCIET|SOCIE|SOCI|SOC)\b.*$', "SOCIETA'", clean_super)
                clean_super = re.sub(r'\b(COOPERATIV|COOPERATI|COOPERAT|COOP)\b.*$', "COOPERATIVA", clean_super)
                current_supermercato = clean_super.strip()
                continue
                
            # 3. CATTURA E ALLINEA I DATI NUMERICI IN BASE ALLA COORDINATA X
            if any(line_text.startswith(prefix) for prefix in ["Quantità", "Prezzo Medio", "Fatturato"]) and current_prodotto and current_supermercato:
                if line_text.startswith("Quantità"):
                    tipo_dato = "Quantità (kg)"
                elif line_text.startswith("Prezzo Medio"):
                    tipo_dato = "Prezzo Medio"
                else:
                    tipo_dato = "Totale (€)"
                
                # Inizializziamo tutti i 13 slot (12 mesi + Totale Anno) a 0.0
                valori_mesi = {m: 0.0 for m in nomi_mesi}
                
                # Analizziamo ogni singola parola/numero della riga dati
                for w in line_words:
                    token = w['text'].strip()
                    # Verifichiamo se il token è un numero (esclude la parola iniziale Quantità/Fatturato/Prezzo)
                    if re.search(r'[\d]', token):
                        x_centro = (w['x0'] + w['x1']) / 2
                        
                        # Definiamo i "bin" (i binari verticali) basati sulle coordinate reali delle colonne nel PDF di Geco
                        # Geco stampa il testo della pagina in un foglio largo circa 842 punti (A4 Orizzontale)
                        # Assegniamo il numero al mese corretto guardando dove cade la sua coordinata X
                        if x_centro < 220:
                            mese_assegnato = "Gennaio"
                        elif x_centro < 265:
                            mese_assegnato = "Febbraio"
                        elif x_centro < 310:
                            mese_assegnato = "Marzo"
                        elif x_centro < 355:
                            mese_assegnato = "Aprile"
                        elif x_centro < 400:
                            mese_assegnato = "Maggio"
                        elif x_centro < 445:
                            mese_assegnato = "Giugno"
                        elif x_centro < 490:
                            mese_assegnato = "Luglio"
                        elif x_centro < 535:
                            mese_assegnato = "Agosto"
                        elif x_centro < 580:
                            mese_assegnato = "Settembre"
                        elif x_centro < 625:
                            mese_assegnato = "Ottobre"
                        elif x_centro < 670:
                            mese_assegnato = "Novembre"
                        elif x_centro < 715:
                            mese_assegnato = "Dicembre"
                        else:
                            mese_assegnato = "Totale Anno"
                        
                        # Pulizia del numero e conversione
                        token_clean = token.replace("€/um", "").replace("kg", "").replace("Kg", "").replace("€", "").strip()
                        token_clean = token_clean.replace(".", "").replace(",", ".")
                        try:
                            val_float = float(token_clean)
                        except:
                            val_float = 0.0
                            
                        # Sommiamo nello slot (gestisce anche i numeri spezzati dal lettore)
                        valori_mesi[mese_assegnato] = val_float
                
                # Appendiamo i record pronti
                for m, val in valori_mesi.items():
                    all_records.append({
                        "Prodotto": current_prodotto,
                        "Supermercato": current_supermercato,
                        "Mese": m,
                        "Tipo_Dato": tipo_dato,
                        "Valore": val
                    })

print("Generazione della tabella finale per Power BI...")
df_flat = pd.DataFrame(all_records)

# Creazione Pivot per affiancare Quantità, Prezzo e Totale
df_final = df_flat.pivot_table(
    index=["Prodotto", "Supermercato", "Mese"],
    columns="Tipo_Dato",
    values="Valore",
    aggfunc="sum"
).reset_index()

# Ordinamento solare dei mesi + Totale Anno
lista_completa_mesi = nomi_mesi
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=lista_completa_mesi, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)

# Ordinamento colonne finale
df_final = df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]

df_final.to_excel(excel_output, index=False)
print(f"File PERFETTO generato con successo! Controlla: {excel_output}")

Analisi chirurgica delle coordinate X del PDF...
Generazione della tabella finale per Power BI...


KeyError: 'Valore'

In [11]:
import pdfplumber
import pandas as pd
import numpy as np
import re

pdf_path = "005)  vendite mensili - raggruppamenti di  prodotto.pdf"
excel_output = "Report_Geco_Finito_PowerBI.xlsx"

all_records = []

# Mappa dei mesi nell'ordine esatto del PDF
nomi_mesi = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
            "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre", "Totale Anno"]

current_prodotto = None
current_supermercato = None

print("Analisi chirurgica delle coordinate X del PDF...")

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        words = page.extract_words(keep_blank_chars=False)
        if not words:
            continue
            
        # Raggruppiamo le parole per riga (coordinata top approssimata a 1 decimale)
        lines = {}
        for w in words:
            top = round(w['top'], 1)
            lines.setdefault(top, []).append(w)
            
        for top in sorted(lines.keys()):
            # Ordiniamo le parole della riga da sinistra a destra
            line_words = sorted(lines[top], key=lambda x: x['x0'])
            line_text = " ".join([w['text'] for w in line_words]).strip()
            
            # FILTRO: Salta intestazioni e l'azienda stessa
            if "Vendita Mensile" in line_text or "Elaborazione" in line_text or "TERRE SABINE" in line_text or "Pagina" in line_text:
                continue
                
            # 1. CATTURA IL PRODOTTO (Riga con Cod: e UM:)
            if "Cod:" in line_text and "UM:" in line_text:
                if not any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "CONSORZIO"]):
                    current_prodotto = line_text
                    current_supermercato = None
                    continue
                    
            # 2. CATTURA IL SUPERMERCATO 
            if any(cli in line_text for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO"]):
                clean_super = line_text.split("Cod:")[0].strip()
                clean_super = re.sub(r'\b(SOCIET|SOCIE|SOCI|SOC)\b.*$', "SOCIETA'", clean_super)
                clean_super = re.sub(r'\b(COOPERATIV|COOPERATI|COOPERAT|COOP)\b.*$', "COOPERATIVA", clean_super)
                current_supermercato = clean_super.strip()
                continue
                
            # 3. CATTURA E ALLINEA I DATI NUMERICI IN BASE ALLA COORDINATA X (Controllo più flessibile)
            is_dati = any(p in line_text for p in ["Quant", "Prezzo", "Fattur"])
            if is_dati and current_prodotto and current_supermercato:
                if "Quant" in line_text:
                    tipo_dato = "Quantità (kg)"
                elif "Prezzo" in line_text:
                    tipo_dato = "Prezzo Medio"
                else:
                    tipo_dato = "Totale (€)"
                
                # Inizializziamo tutti i 13 slot (12 mesi + Totale Anno) a 0.0
                valori_mesi = {m: 0.0 for m in nomi_mesi}
                
                # Analizziamo ogni singola parola/numero della riga dati
                for w in line_words:
                    token = w['text'].strip()
                    # Verifichiamo se il token contiene un numero (esclude le parole di testo iniziali)
                    if re.search(r'[\d]', token):
                        x_centro = (w['x0'] + w['x1']) / 2
                        
                        # Assegniamo il numero al mese corretto guardando dove cade la sua coordinata X
                        if x_centro < 220:
                            mese_assegnato = "Gennaio"
                        elif x_centro < 265:
                            mese_assegnato = "Febbraio"
                        elif x_centro < 310:
                            mese_assegnato = "Marzo"
                        elif x_centro < 355:
                            mese_assegnato = "Aprile"
                        elif x_centro < 400:
                            mese_assegnato = "Maggio"
                        elif x_centro < 445:
                            mese_assegnato = "Giugno"
                        elif x_centro < 490:
                            mese_assegnato = "Luglio"
                        elif x_centro < 535:
                            mese_assegnato = "Agosto"
                        elif x_centro < 580:
                            mese_assegnato = "Settembre"
                        elif x_centro < 625:
                            mese_assegnato = "Ottobre"
                        elif x_centro < 670:
                            mese_assegnato = "Novembre"
                        elif x_centro < 715:
                            mese_assegnato = "Dicembre"
                        else:
                            mese_assegnato = "Totale Anno"
                        
                        # Pulizia del numero e conversione
                        token_clean = token.replace("€/um", "").replace("kg", "").replace("Kg", "").replace("€", "").strip()
                        token_clean = token_clean.replace(".", "").replace(",", ".")
                        try:
                            val_float = float(token_clean)
                        except:
                            val_float = 0.0
                            
                        valori_mesi[mese_assegnato] = val_float
                
                # Appendiamo i record pronti
                for m, val in valori_mesi.items():
                    all_records.append({
                        "Prodotto": current_prodotto,
                        "Supermercato": current_supermercato,
                        "Mese": m,
                        "Tipo_Dato": tipo_dato,
                        "Valore": val
                    })

print(f"Righe dati intercettate: {len(all_records)}. Generazione tabella...")

if len(all_records) == 0:
    print("ERRORE: Nessun dato estratto. Controlla che i nomi dei prodotti o supermercati nel PDF coincidano.")
else:
    df_flat = pd.DataFrame(all_records)

    # Creazione Pivot per affiancare Quantità, Prezzo e Totale
    df_final = df_flat.pivot_table(
        index=["Prodotto", "Supermercato", "Mese"],
        columns="Tipo_Dato",
        values="Valore",
        aggfunc="sum"
    ).reset_index()

    # Ordinamento solare dei mesi + Totale Anno
    df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=nomi_mesi, ordered=True)
    df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)

    # Ordinamento colonne finale
    df_final = df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]

    df_final.to_excel(excel_output, index=False)
    print(f"File PERFETTO generato con successo! Controlla: {excel_output}")

Analisi chirurgica delle coordinate X del PDF...
Righe dati intercettate: 0. Generazione tabella...
ERRORE: Nessun dato estratto. Controlla che i nomi dei prodotti o supermercati nel PDF coincidano.


In [13]:
!pip install xlrd openpyxl

import pandas as pd
import numpy as np

# Carica il file Excel originale esportato da Geco
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto.xls"
file_output_powerbi = "Dati_Pesche_Pronti_PowerBI.xlsx"

# Leggiamo il file saltando le righe vuote iniziali, usando la riga dei mesi come intestazione
df = pd.read_excel(file_geco, header=5)

# Pulizia colonne: teniamo solo quelle utili ed eliminiamo quelle completamente vuote
df = df.dropna(how='all')

all_data = []
current_prodotto = None
current_supermercato = None

# Lista ordinata dei mesi come appaiono nelle colonne del file di Geco
mesi_colonne = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
                "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre", "Totale Anno"]

# Troviamo gli indici esatti delle colonne dei mesi nel file Excel
colonne_esistenti = df.columns.tolist()
indici_mesi = {}
for mese in mesi_colonne:
    for idx, col in enumerate(colonne_esistenti):
        if str(col).strip() == mese:
            indici_mesi[mese] = idx
            break

# Scansione riga per riga
for index, row in df.iterrows():
    valori_riga = row.values.tolist()
    riga_str = " ".join([str(v) for v in valori_riga if pd.notna(v)])
    
    if not riga_str:
        continue
        
    # 1. Identifica il Prodotto (es. PESCHE BIANCHE 1 Kg - Cod:FPEB07)
    if "Cod:" in riga_str and "UM:" in riga_str:
        if not any(cli in riga_str for cli in ["SOCIETA", "SRL", "COOP", "CONSORZIO"]):
            # Cerca la cella che contiene il nome del prodotto
            for v in valori_riga:
                if pd.notna(v) and "Cod:" in str(v):
                    current_prodotto = str(v).strip()
                    break
            current_supermercato = None
            continue
            
    # 2. Identifica il Supermercato / Cliente
    if any(cli in riga_str for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO"]):
        for v in valori_riga:
            if pd.notna(v) and any(cli in str(v) for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA"]):
                current_supermercato = str(v).strip()
                break
        continue
        
    # 3. Estrazione dei dati (Quantità, Prezzo Medio, Fatturato)
    # Nel file XLS di Geco, le righe numeriche si riconoscono perché hanno i numeri posizionati sotto le colonne dei mesi
    if current_prodotto and current_supermercato:
        # Capiamo che tipo di riga è guardando i valori presenti
        # Geco mette 3 righe consecutive per ogni cliente: Quantità (interi), Fatturato (decimali), Prezzo (medie)
        # Per non sbagliare, contiamo quante righe abbiamo già estratto per questo blocco
        
        # Estraiamo i valori per ogni mese usando la posizione esatta della colonna blindata
        ha_numeri = False
        valori_mesi = {}
        
        for mese, col_idx in indici_mesi.items():
            val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
            val_float = 0.0
            if pd.notna(val):
                try:
                    val_float = float(str(val).replace(".", "").replace(",", "."))
                    if val_float != 0:
                        ha_numeri = True
                except:
                    val_float = 0.0
            valori_mesi[mese] = val_float
            
        if ha_numeri:
            # Determina il tipo dato dalla sequenza o da parole chiave residue
            # Creiamo un record pulito in verticale per Power BI
            for mese, val_float in valori_mesi.items():
                all_data.append({
                    "Prodotto": current_prodotto,
                    "Supermercato": current_supermercato,
                    "Mese": mese,
                    "Valore": val_float,
                    # Temporaneo per fare la pivot successiva
                    "Fila": index 
                })

# Ricostruzione Tabella Ordinata per Power BI
df_flat = pd.DataFrame(all_data)

# Poiché Geco scrive 3 righe sotto ogni supermercato, le identifichiamo in ordine: 
# riga 1 = Quantità, riga 2 = Fatturato, riga 3 = Prezzo Medio
df_flat['Tipo_Dato'] = df_flat.groupby(['Prodotto', 'Supermercato', 'Mese']).cumcount()
df_flat['Tipo_Dato'] = df_flat['Tipo_Dato'].map({0: "Quantità (kg)", 1: "Totale (€)", 2: "Prezzo Medio"})

# Facciamo il Pivot finale
df_final = df_flat.pivot_table(
    index=["Prodotto", "Supermercato", "Mese"],
    columns="Tipo_Dato",
    values="Valore",
    aggfunc="first"
).reset_index()

# Ordinamento Cronologico
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_colonne, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)

# Riordino colonne
df_final = df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]

# Salvataggio
df_final.to_excel(file_output_powerbi, index=False)
print(f"Fatto! File pulito e raddrizzato salvato in: {file_output_powerbi}")

KeyError: 'Prodotto'

In [15]:

import pandas as pd
import numpy as np

# Carica il file Excel originale esportato da Geco
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto.xls"
file_output_powerbi = "Dati_Pesche_Pronti_PowerBI.xlsx"

# Proviamo a leggerlo specificando il motore moderno openpyxl o quello standard
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Pulizia colonne: eliminiamo le righe completamente vuote
df = df.dropna(how='all')

all_data = []
current_prodotto = "PRODOTTO GENERALE"  # Valore di backup
current_supermercato = "CLIENTE GENERALE" # Valore di backup

# Lista ordinata dei mesi come appaiono nelle colonne del file di Geco
mesi_colonne = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
                "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre", "Totale Anno"]

# Troviamo gli indici esatti delle colonne dei mesi nel file Excel
colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {}
for mese in mesi_colonne:
    for idx, col in enumerate(colonne_esistenti):
        if mese in col:
            indici_mesi[mese] = idx
            break

# Scansione riga per riga
for index, row in df.iterrows():
    valori_riga = row.values.tolist()
    
    # Uniamo tutti i testi della riga per capire cosa contiene
    riga_str = " ".join([str(v) for v in valori_riga if pd.notna(v)]).strip()
    
    if not riga_str or "TERRE SABINE" in riga_str or "Totali" in riga_str:
        continue
        
    # 1. Identifica il Prodotto (es. se contiene Cod: e UM: ma NON i nomi dei clienti)
    if "Cod:" in riga_str and "UM:" in riga_str:
        if not any(cli in riga_str for cli in ["SOCIETA", "SRL", "COOP", "CONSORZIO", "UNICOOP"]):
            current_prodotto = riga_str
            current_supermercato = "CE.DI.GROS SOCIETA'" # Se l'estrazione è solo per Cedigros, lo forziamo come salvagente
            continue
            
    # 2. Identifica il Supermercato / Cliente (se presente in riga)
    if any(cli in riga_str for cli in ["SOCIETA", "SRL", "COOP", "MARKET", "SPA", "S.C.", "CONSORZIO", "CENTRO", "UNICOOP"]):
        current_supermercato = riga_str
        continue
        
    # 3. Estrazione dei dati numerici usando gli indici blindati delle colonne
    valori_mesi = {}
    ha_numeri = False
    
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val):
            try:
                # Pulizia da eventuali spazi o simboli residui
                val_clean = str(val).replace(" ", "").replace("€", "").replace("kg", "")
                val_float = float(val_clean)
                if val_float != 0:
                    ha_numeri = True
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
        
    # Se la riga conteneva effettivamente dei dati commerciali, la salviamo
    if ha_numeri:
        for mese, val_float in valori_mesi.items():
            all_data.append({
                "Prodotto": current_prodotto,
                "Supermercato": current_supermercato,
                "Mese": mese,
                "Valore": val_float
            })

# Controllo di sicurezza prima di fare operazioni su DataFrame
if not all_data:
    print("ERRORE: Non è stato possibile estrarre i dati. Verifica il nome delle colonne del file XLS.")
else:
    df_flat = pd.DataFrame(all_data)
    
    # Identifichiamo la sequenza delle 3 righe (Quantità, Totale, Prezzo Medio) per ogni blocco
    df_flat['Tipo_Dato'] = df_flat.groupby(['Prodotto', 'Supermercato', 'Mese']).cumcount()
    df_flat['Tipo_Dato'] = df_flat['Tipo_Dato'].map({0: "Quantità (kg)", 1: "Totale (€)", 2: "Prezzo Medio"})
    
    # Pivot finale per Power BI
    df_final = df_final = df_flat.pivot_table(
        index=["Prodotto", "Supermercato", "Mese"],
        columns="Tipo_Dato",
        values="Valore",
        aggfunc="first"
    ).reset_index()
    
    # Ordinamento cronologico corretto
    df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_colonne, ordered=True)
    df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
    
    # Riordino Colonne
    df_final = df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]
    
    # Salvataggio
    df_final.to_excel(file_output_powerbi, index=False)
    print(f"SÌ! File raddrizzato con successo e pronto per Power BI: {file_output_powerbi}")

SÌ! File raddrizzato con successo e pronto per Power BI: Dati_Pesche_Pronti_PowerBI.xlsx


In [16]:
import pandas as pd
import numpy as np

# File originali
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto.xls"
file_output_powerbi = "Dati_Pesche_Pronti_PowerBI.xlsx"

# Lettura file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista ordinata dei mesi del file
mesi_colonne = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
                "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre", "Totale Anno"]

# Trova indici delle colonne dei mesi
colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_colonne for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []

# Trasformiamo il dataframe in una lista di righe per muoverci avanti e indietro facilmente
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    """Estrae i valori numerici per ogni mese usando gli indici fissi delle colonne"""
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val):
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione chirurgica basata sulla posizione del Prodotto
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    # Quando troviamo la riga del Prodotto (quella centrale del blocco)
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = "CE.DI.GROS SOCIETA'" # Forzato fisso come richiesto dall'analisi
        
        # 1. LA RIGA SOPRA (idx - 1) è sempre il TOTALE IN EURO
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            for mese, val in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": mese, "Tipo_Dato": "Totale (€)", "Valore": val})
                
        # 2. LA RIGA SOTTO (idx + 1) è sempre la QUANTITÀ (KG)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            for mese, val in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": mese, "Tipo_Dato": "Quantità (kg)", "Valore": val})
                
        # 3. LA RIGA ANCORA SOTTO (idx + 2) è sempre il PREZZO MEDIO
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            for mese, val in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": mese, "Tipo_Dato": "Prezzo Medio", "Valore": val})

# Creazione del DataFrame finale raddrizzato
if not all_data:
    print("Errore nell'estrazione. Verifica la struttura del file.")
else:
    df_flat = pd.DataFrame(all_data)
    
    # Facciamo la pivot per mettere le tre metriche su colonne separate
    df_final = df_flat.pivot_table(
        index=["Prodotto", "Supermercato", "Mese"],
        columns="Tipo_Dato",
        values="Valore",
        aggfunc="first"
    ).reset_index()
    
    # Ordinamento cronologico dei mesi
    df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_colonne, ordered=True)
    df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
    
    # Ordinamento colonne desiderato
    df_final = df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]
    
    # Salvataggio file
    df_final.to_excel(file_output_powerbi, index=False)
    print(f"Raddrizzamento completato! Ora Maggio avrà: Totale € = 4628, KG = 1780, Prezzo = 2.6. Controlla il file: {file_output_powerbi}")

Raddrizzamento completato! Ora Maggio avrà: Totale € = 4628, KG = 1780, Prezzo = 2.6. Controlla il file: Dati_Pesche_Pronti_PowerBI.xlsx


In [17]:
import pandas as pd
import numpy as np

# File originali
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto.xls"
file_output_powerbi = "Dati_Pesche_Pronti_PowerBI.xlsx"

# Lettura file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista ordinata dei 12 mesi solari
mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

# Trova indici delle colonne dei 12 mesi
colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    """Estrae i valori numerici per i 12 mesi usando gli indici fissi delle colonne"""
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val):
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione basata sulla posizione del Prodotto
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = "CE.DI.GROS SOCIETA'"
        
        # 1. TOTALE IN EURO (Riga Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values()) # Calcolo matematico diretto
            
            for mese, val in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": mese, "Tipo_Dato": "Totale (€)", "Valore": val})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. QUANTITÀ KG (Riga Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values()) # Calcolo matematico diretto
            
            for mese, val in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": mese, "Tipo_Dato": "Quantità (kg)", "Valore": val})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. PREZZO MEDIO (Riga Ancora Sotto)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            # Il prezzo medio annuo non è la somma dei prezzi, ma il Fatturato Totale / KG Totali
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            
            for mese, val in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": mese, "Tipo_Dato": "Prezzo Medio", "Valore": val})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Creazione del DataFrame finale
if not all_data:
    print("Errore nell'estrazione. Verifica la struttura.")
else:
    df_flat = pd.DataFrame(all_data)
    
    df_final = df_flat.pivot_table(
        index=["Prodotto", "Supermercato", "Mese"],
        columns="Tipo_Dato",
        values="Valore",
        aggfunc="first"
    ).reset_index()
    
    # Ordinamento cronologico inclusa la riga Totale Anno in fondo
    mesi_ordinamento = mesi_solari + ["Totale Anno"]
    df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
    df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
    
    df_final = df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]
    
    df_final.to_excel(file_output_powerbi, index=False)
    print(f"Fatto! Ora anche la riga 'Totale Anno' è calcolata e compilata. Controlla: {file_output_powerbi}")

Fatto! Ora anche la riga 'Totale Anno' è calcolata e compilata. Controlla: Dati_Pesche_Pronti_PowerBI.xlsx


In [19]:
import pandas as pd
import numpy as np

# Inserisci qui il nome esatto del tuo file delle ciliegie scaricato da Geco
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto cili.xls"
file_output_powerbi = "Dati_Ciliegie_Pronti_PowerBI.xlsx"

# Lettura file (con gestione del motore di lettura)
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista ordinata dei 12 mesi solari
mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

# Trova gli indici delle colonne dei 12 mesi
colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    """Estrae i valori numerici per i 12 mesi usando gli indici fissi delle colonne"""
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione chirurgica basata sulle righe
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    # Quando intercetta la riga centrale del prodotto
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = "CE.DI.GROS SOCIETA'"
        
        # 1. TOTALE IN EURO (Riga Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for mese, val in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": mese, "Tipo_Dato": "Totale (€)", "Valore": val})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. QUANTITÀ KG (Riga Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for mese, val in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": mese, "Tipo_Dato": "Quantità (kg)", "Valore": val})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. PREZZO MEDIO (Riga Ancora Sotto)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for mese, val in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": mese, "Tipo_Dato": "Prezzo Medio", "Valore": val})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Creazione del DataFrame finale raddrizzato
if all_data:
    df_flat = pd.DataFrame(all_data)
    df_final = df_flat.pivot_table(
        index=["Prodotto", "Supermercato", "Mese"],
        columns="Tipo_Dato",
        values="Valore",
        aggfunc="first"
    ).reset_index()
    
    # Ordinamento dei mesi e salvataggio
    mesi_ordinamento = mesi_solari + ["Totale Anno"]
    df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
    df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
    df_final = df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]
    
    df_final.to_excel(file_output_powerbi, index=False)
    print(f"File raddrizzato con successo: {file_output_powerbi}")

File raddrizzato con successo: Dati_Ciliegie_Pronti_PowerBI.xlsx


In [20]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER OGNI FRUTTO
# ==========================================================
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto albicocche.xls" 
file_output_powerbi = "Dati_Albicocche_Pronti.xlsx"
# ==========================================================

# (Il resto del codice resta invariato, lo incollo qui per comodità)
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = "CE.DI.GROS SOCIETA'"
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)
print("File esportato con successo!")

File esportato con successo!


In [21]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER OGNI FRUTTO
# ==========================================================
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto tirreno cili.xls" 
file_output_powerbi = "Dati_Albicocche_Pronti.xlsx"
# ==========================================================

# (Il resto del codice resta invariato, lo incollo qui per comodità)
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = "CE.DI.GROS SOCIETA'"
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)
print("File esportato con successo!")

File esportato con successo!


In [23]:

import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA I NOMI DEI FILE PER TIRRENOFRUIT
# ==========================================================
file_geco = "Fine master/005)  vendite mensili - raggruppamenti di  prodotto tirreno pesche.xls" # <--- Inserisci il nome del file
file_output_powerbi = "Dati_Tirrenofruit_ProntiPesche.xlsx"
# ==========================================================

# Identificativo supermercato
NOME_SUPERMERCATO = "TIRRENOFRUIT"

# Caricamento file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista mesi
mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)
print(f"File per {NOME_SUPERMERCATO} esportato con successo!")

File per TIRRENOFRUIT esportato con successo!


In [27]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA I NOMI DEI FILE PER TIRRENOFRUIT
# ==========================================================
file_geco = "Fine master/005)  vendite mensili - raggruppamenti di  prodotto tirreno albi.xls" # <--- Inserisci il nome del file
file_output_powerbi = "Dati_Tirrenofruit_ProntiAlbi.xlsx"
# ==========================================================

# Identificativo supermercato
NOME_SUPERMERCATO = "TIRRENOFRUIT"

# Caricamento file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista mesi
mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)
print(f"File per {NOME_SUPERMERCATO} esportato con successo!")

File per TIRRENOFRUIT esportato con successo!


In [28]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA I NOMI DEI FILE PER TIRRENOFRUIT
# ==========================================================
file_geco = "Fine master/005)  vendite mensili - raggruppamenti di  prodotto tirreno cili.xls" # <--- Inserisci il nome del file
file_output_powerbi = "Dati_Tirrenofruit_ProntiCili.xlsx"
# ==========================================================

# Identificativo supermercato
NOME_SUPERMERCATO = "TIRRENOFRUIT"

# Caricamento file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista mesi
mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)
print(f"File per {NOME_SUPERMERCATO} esportato con successo!")

File per TIRRENOFRUIT esportato con successo!


In [29]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER UNICOOP
# ==========================================================
file_geco = "Fine master/005)  vendite mensili - raggruppamenti di  prodotto albi coop unicoop.xls" # <--- Inserisci il nome del file
file_output_powerbi = "Dati_Unicoop_ProntiAlbi.xlsx"
# ==========================================================

NOME_SUPERMERCATO = "UNICOOP"

# Caricamento file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista mesi
mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)
print(f"File per {NOME_SUPERMERCATO} esportato con successo!")

File per UNICOOP esportato con successo!


In [31]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER UNICOOP
# ==========================================================
file_geco = "Fine master/005)  vendite mensili - raggruppamenti di  prodotto pesche coop unicoop.xls" # <--- Inserisci il nome del file
file_output_powerbi = "Dati_UnicoopEtruria_ProntePesche.xlsx"
# ==========================================================

NOME_SUPERMERCATO = "UNICOOP Etruria"

# Caricamento file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista mesi
mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)
print(f"File per {NOME_SUPERMERCATO} esportato con successo!")

File per UNICOOP Etruria esportato con successo!


In [32]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER UNICOOP
# ==========================================================
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto coop albi.xls" # <--- Inserisci il nome del file
file_output_powerbi = "Dati_COOP_PronteAlbi.xlsx"
# ==========================================================

NOME_SUPERMERCATO = "COOP"

# Caricamento file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista mesi
mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)
print(f"File per {NOME_SUPERMERCATO} esportato con successo!")

File per COOP esportato con successo!


In [33]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER UNICOOP
# ==========================================================
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto coop cili.xls" # <--- Inserisci il nome del file
file_output_powerbi = "Dati_COOP_PronteCili.xlsx"
# ==========================================================

NOME_SUPERMERCATO = "COOP"

# Caricamento file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista mesi
mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)
print(f"File per {NOME_SUPERMERCATO} esportato con successo!")

File per COOP esportato con successo!


In [37]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER UNICOOP
# ==========================================================
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto unicoop pesche.xls" # <--- Inserisci il nome del file
file_output_powerbi = "Dati_UNICOOP_ProntePesche.xlsx"
# ==========================================================

NOME_SUPERMERCATO = "UNICOOP"

# Caricamento file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

# Lista mesi
mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)
print(f"File per {NOME_SUPERMERCATO} esportato con successo!")

File per UNICOOP esportato con successo!


In [41]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER L'ANNO 2024
# ==========================================================
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto albi coop 2024.xls"  # <--- Inserisci il nome del file 2024
file_output_powerbi = "Dati_Coop_2024_ProntiAlbi.xlsx"
# ==========================================================

NOME_SUPERMERCATO = "COOP"

# Caricamento file (identico a prima)
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe (logica Sopra / Centro / Sotto)
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)

print(f"File {NOME_SUPERMERCATO} elaborato correttamente!")

File COOP elaborato correttamente!


In [42]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER L'ANNO 2024
# ==========================================================
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto cili coop 2024.xls"  # <--- Inserisci il nome del file 2024
file_output_powerbi = "Dati_Coop_2024_ProntiCili.xlsx"
# ==========================================================

NOME_SUPERMERCATO = "COOP"

# Caricamento file (identico a prima)
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe (logica Sopra / Centro / Sotto)
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)

print(f"File {NOME_SUPERMERCATO} elaborato correttamente!")

File COOP elaborato correttamente!


In [43]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER L'ANNO 2024
# ==========================================================
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto pesche coop 2024.xls"  # <--- Inserisci il nome del file 2024
file_output_powerbi = "Dati_Coop_2024_ProntiPesche.xlsx"
# ==========================================================

NOME_SUPERMERCATO = "COOP"

# Caricamento file (identico a prima)
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe (logica Sopra / Centro / Sotto)
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro (Sopra)
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG (Sotto)
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio (Sotto ancora)
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Mese"]).reset_index(drop=True)
df_final[["Prodotto", "Supermercato", "Mese", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]].to_excel(file_output_powerbi, index=False)

print(f"File {NOME_SUPERMERCATO} elaborato correttamente!")

File COOP elaborato correttamente!


In [56]:
import pandas as pd
import numpy as np

# ==========================================================
# CAMBIA IL NOME DEL FILE PER L'ANNO 2024
# ==========================================================
file_geco = "005)  vendite mensili - raggruppamenti di  prodotto pesche tirreno 2023.xls"  # <--- INSERISCI QUI IL NOME
file_output_powerbi = "Dati_Tirrenofruit_2023_ProntiPesche.xlsx"
# ==========================================================

NOME_SUPERMERCATO = "TIRRENOFRUIT"
ANNO_RIFERIMENTO = 2023

# Lettura file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe con inclusione dell'Anno
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione con colonna Anno inclusa
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese", "Anno"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Anno", "Mese"]).reset_index(drop=True)
df_final = df_final[["Prodotto", "Supermercato", "Mese", "Anno", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]

df_final.to_excel(file_output_powerbi, index=False)
print(f"File per {NOME_SUPERMERCATO} {ANNO_RIFERIMENTO} esportato correttamente con la colonna Anno!")

File per TIRRENOFRUIT 2023 esportato correttamente con la colonna Anno!


In [65]:
import pandas as pd
import numpy as np

file_geco = "005)  vendite mensili - raggruppamenti di  prodotto pesche tirreno 2022.xls"  # <--- INSERISCI QUI IL NOME
file_output_powerbi = "Dati_IRRENOFRUIT_2022_ProntiPesche.xlsx"
# ==========================================================

NOME_SUPERMERCATO = "TIRRENOFRUIT"
ANNO_RIFERIMENTO = 2022

# Lettura file
try:
    df = pd.read_excel(file_geco, header=5, engine='openpyxl')
except:
    df = pd.read_excel(file_geco, header=5)

mesi_solari = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", 
               "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "Dicembre"]

colonne_esistenti = [str(c).strip() for c in df.columns.tolist()]
indici_mesi = {mese: idx for mese in mesi_solari for idx, col in enumerate(colonne_esistenti) if mese in col}

all_data = []
righe = df.values.tolist()

def estrai_numeri_riga(valori_riga):
    valori_mesi = {}
    for mese, col_idx in indici_mesi.items():
        val = valori_riga[col_idx] if col_idx < len(valori_riga) else np.nan
        val_float = 0.0
        if pd.notna(val) and str(val).strip() != "":
            try:
                val_float = float(str(val).replace(" ", "").replace("€", "").replace("kg", ""))
            except:
                val_float = 0.0
        valori_mesi[mese] = val_float
    return valori_mesi

# Scansione righe con inclusione dell'Anno
for idx in range(len(righe)):
    valori_attuali = righe[idx]
    riga_str = " ".join([str(v) for v in valori_attuali if pd.notna(v)]).strip()
    
    if "Cod:" in riga_str and "UM:" in riga_str:
        current_prodotto = riga_str
        current_supermercato = NOME_SUPERMERCATO
        
        # 1. Totale Euro
        if idx - 1 >= 0:
            mesi_euro = estrai_numeri_riga(righe[idx - 1])
            totale_anno_euro = sum(mesi_euro.values())
            for m, v in mesi_euro.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Totale (€)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Totale (€)", "Valore": totale_anno_euro})
                
        # 2. Quantità KG
        if idx + 1 < len(righe):
            mesi_kg = estrai_numeri_riga(righe[idx + 1])
            totale_anno_kg = sum(mesi_kg.values())
            for m, v in mesi_kg.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Quantità (kg)", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Quantità (kg)", "Valore": totale_anno_kg})
                
        # 3. Prezzo Medio
        if idx + 2 < len(righe):
            mesi_prezzo = estrai_numeri_riga(righe[idx + 2])
            totale_anno_prezzo = totale_anno_euro / totale_anno_kg if totale_anno_kg != 0 else 0.0
            for m, v in mesi_prezzo.items():
                all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": m, "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Prezzo Medio", "Valore": v})
            all_data.append({"Prodotto": current_prodotto, "Supermercato": current_supermercato, "Mese": "Totale Anno", "Anno": ANNO_RIFERIMENTO, "Tipo_Dato": "Prezzo Medio", "Valore": round(totale_anno_prezzo, 2)})

# Finalizzazione con colonna Anno inclusa
df_final = pd.DataFrame(all_data).pivot_table(index=["Prodotto", "Supermercato", "Mese", "Anno"], columns="Tipo_Dato", values="Valore", aggfunc="first").reset_index()
mesi_ordinamento = mesi_solari + ["Totale Anno"]
df_final["Mese"] = pd.Categorical(df_final["Mese"], categories=mesi_ordinamento, ordered=True)
df_final = df_final.sort_values(by=["Prodotto", "Supermercato", "Anno", "Mese"]).reset_index(drop=True)
df_final = df_final[["Prodotto", "Supermercato", "Mese", "Anno", "Quantità (kg)", "Prezzo Medio", "Totale (€)"]]

df_final.to_excel(file_output_powerbi, index=False)
print(f"File per {NOME_SUPERMERCATO} {ANNO_RIFERIMENTO} esportato correttamente con la colonna Anno!")

File per TIRRENOFRUIT 2022 esportato correttamente con la colonna Anno!


In [ ]:
import pandas as pd
import glob
import os

# Definiamo dove cercare: "**/*" significa "cerca in questa cartella e in tutte le sottocartelle"
pattern = "**/Dati_*_Pronti*.xlsx"

# 1. Trova tutti i file che rispettano il pattern
lista_file = glob.glob(pattern, recursive=True)

if not lista_file:
    print("Nessun file trovato! Assicurati che i file terminino con '_Pronti*.xlsx'")
else:
    tutti_i_dati = []
    
    print(f"Trovati {len(lista_file)} file. Inizio elaborazione...")
    
    # 2. Leggi ogni file e aggiungilo alla lista
    for file in lista_file:
        try:
            df_temp = pd.read_excel(file)
            tutti_i_dati.append(df_temp)
            print(f"-> Aggiunto: {file}")
        except Exception as e:
            print(f"Errore nella lettura del file {file}: {e}")
    
    # 3. Unisci tutto
    if tutti_i_dati:
        df_finale = pd.concat(tutti_i_dati, ignore_index=True)
        
        # 4. Salva il database completo nella cartella principale
        nome_output = "Database_Vendite_Completo.xlsx"
        df_finale.to_excel(nome_output, index=False)
        
        print("-" * 30)
        print(f"DATABASE COMPLETATO CON SUCCESSO!")
        print(f"Totale righe nel database: {len(df_finale)}")
        print(f"File salvato come: {nome_output}")
    else:
        print("Nessun dato è stato caricato correttamente.")

In [2]:
import pandas as pd
import glob
import os

# Definiamo il nome della cartella specifica
cartella_target = "/Users/gretamerzetti/Desktop/FineMaster"

# Il pattern ora dice: "vai dentro finemaster e cerca ovunque (**/*) 
# tutti i file che iniziano con Dati_ e finiscono con _Pronti.xlsx"
pattern = os.path.join(cartella_target, "**", "Dati_*_Pronti*.xlsx")

# 1. Trova tutti i file
lista_file = glob.glob(pattern, recursive=True)

if not lista_file:
    print(f"Nessun file trovato dentro la cartella '{cartella_target}'!")
    print("Controlla che la cartella sia nella stessa posizione dello script.")
else:
    tutti_i_dati = []
    
    print(f"Trovati {len(lista_file)} file in '{cartella_target}'. Elaborazione...")
    
    for file in lista_file:
        try:
            df_temp = pd.read_excel(file)
            tutti_i_dati.append(df_temp)
            print(f"-> Aggiunto: {file}")
        except Exception as e:
            print(f"Errore nel file {file}: {e}")
    
    # 2. Unisci e salva
    if tutti_i_dati:
        df_finale = pd.concat(tutti_i_dati, ignore_index=True)
        nome_output = "Database_Vendite_Completo.xlsx"
        df_finale.to_excel(nome_output, index=False)
        
        print("-" * 30)
        print(f"DATABASE COMPLETATO!")
        print(f"Totale righe: {len(df_finale)}")
        print(f"File salvato: {nome_output}")

Trovati 42 file in '/Users/gretamerzetti/Desktop/FineMaster'. Elaborazione...
-> Aggiunto: /Users/gretamerzetti/Desktop/FineMaster/2022/Dati_coop_2022_ProntiCili.xlsx
-> Aggiunto: /Users/gretamerzetti/Desktop/FineMaster/2022/Dati_CEDIGROS_2022_ProntiAlbi.xlsx
-> Aggiunto: /Users/gretamerzetti/Desktop/FineMaster/2022/Dati_Coop_2022_ProntiAlbi.xlsx
-> Aggiunto: /Users/gretamerzetti/Desktop/FineMaster/2022/Dati_CEDIGROS_2022_ProntiCili.xlsx
-> Aggiunto: /Users/gretamerzetti/Desktop/FineMaster/2022/Dati_TIRRENOFRUIT_2022_ProntiPesche.xlsx
-> Aggiunto: /Users/gretamerzetti/Desktop/FineMaster/2022/Dati_CEDIGROS_2022_ProntiPesche.xlsx
-> Aggiunto: /Users/gretamerzetti/Desktop/FineMaster/2022/Dati_TIRRENOFRUIT_2022_ProntiCili.xlsx
-> Aggiunto: /Users/gretamerzetti/Desktop/FineMaster/2022/Dati_TIRRENOFRUIT_2022_ProntiAlbi.xlsx
-> Aggiunto: /Users/gretamerzetti/Desktop/FineMaster/2022/Dati_coop_2022_ProntiPesche.xlsx
-> Aggiunto: /Users/gretamerzetti/Desktop/FineMaster/2025/Dati_Cedigros_ProntiP

In [3]:
import pandas as pd

df = pd.read_excel("Database_Vendite_Completo.xlsx")

dim_prodotti = df[['Prodotto']].drop_duplicates().reset_index(drop=True)
dim_prodotti.to_excel("Dim_Prodotti.xlsx", index=False)

dim_supermercati = df[['Supermercato']].drop_duplicates().reset_index(drop=True)
dim_supermercati.to_excel("Dim_Supermercati.xlsx", index=False)

print("Tabelle Dimensione create con successo!")

Tabelle Dimensione create con successo!
